# Building with the Claude API

Interactive companion notebook for [Anthropic Skilljar Course #287725](https://anthropic.skilljar.com/claude-with-the-anthropic-api/287725).

## What you'll learn
1. **Setup & authentication** — installing the SDK, managing API keys safely
2. **Basic messages** — your first API call, parsing responses
3. **System prompts** — steering Claude's behavior
4. **Multi-turn conversations** — managing chat history (the API is stateless!)
5. **Streaming** — real-time token-by-token output
6. **Model selection** — Opus vs Sonnet vs Haiku tradeoffs
7. **Token counting** — estimating cost before you call

## Prerequisites
- An Anthropic API key from [console.anthropic.com](https://console.anthropic.com/)
- The `anthropic` Python SDK (already installed in this venv)

---
## 1. Setup & Authentication

The SDK reads the API key from the `ANTHROPIC_API_KEY` environment variable by default. **Never hardcode keys in notebooks** — they get committed to git.

**Option A (recommended):** export the key in your shell before launching Jupyter:
```bash
export ANTHROPIC_API_KEY="sk-ant-..."
```

**Option B:** set it just for this kernel session via `os.environ` (it won't persist outside this notebook):

In [ ]:
import os
import getpass

# Only prompt if not already set in the environment
if not os.environ.get('ANTHROPIC_API_KEY'):
    os.environ['ANTHROPIC_API_KEY'] = getpass.getpass('Enter your Anthropic API key: ')

print(f"Key loaded: {os.environ['ANTHROPIC_API_KEY'][:10]}...{os.environ['ANTHROPIC_API_KEY'][-4:]}")

In [ ]:
# Verify the SDK is installed and create the client
import anthropic

client = anthropic.Anthropic()  # auto-reads ANTHROPIC_API_KEY
print(f'anthropic SDK version: {anthropic.__version__}')
print(f'Client ready: {client}')

---
## 2. Your First Message

Every API call uses the same endpoint: `client.messages.create()`. Three required parameters:

- `model` — which Claude to use (e.g. `claude-haiku-4-5` for fast/cheap)
- `max_tokens` — hard cap on the response length
- `messages` — the conversation, alternating between `user` and `assistant`

### Why `max_tokens`?
Unlike many APIs, this is **required**. Claude won't ramble forever — it stops when it's done OR when this cap is hit (whichever comes first). For chat-style outputs, `1024`–`16000` is typical.

In [ ]:
response = client.messages.create(
    model='claude-haiku-4-5',  # cheapest/fastest — great for learning
    max_tokens=1024,
    messages=[
        {'role': 'user', 'content': 'In one sentence, what is the Anthropic API?'}
    ]
)

# response.content is a LIST of content blocks (not a string!)
# Each block has a .type — for text responses, it's 'text'
for block in response.content:
    if block.type == 'text':
        print(block.text)

# Inspect the metadata
print(f'\n--- metadata ---')
print(f'stop_reason: {response.stop_reason}')
print(f'input tokens:  {response.usage.input_tokens}')
print(f'output tokens: {response.usage.output_tokens}')

### Why `response.content` is a list (not a string)

Claude can return **multiple content blocks** in one response — a thinking block followed by text, or a text explanation followed by a tool call. Always check `block.type` before accessing `.text`. This is a frequent gotcha when extended thinking is enabled.

### `stop_reason` values
| Value | Meaning |
|---|---|
| `end_turn` | Claude finished naturally ✓ |
| `max_tokens` | Hit the cap — increase `max_tokens` |
| `tool_use` | Claude wants to call a tool — execute and continue |
| `refusal` | Claude declined (safety) |

---
## 3. System Prompts — Steering Behavior

The `system` parameter sets Claude's persona, role, and constraints. It's not part of the `messages` array — it's a separate top-level field.

**Same user message, different system prompts → very different answers:**

In [ ]:
user_question = 'What is recursion?'

personas = {
    'pirate': 'You are a pirate. Answer all questions in pirate speak. Keep answers under 3 sentences.',
    'professor': 'You are a CS professor. Answer with formal precision and a Big-O analysis when relevant. Keep answers under 3 sentences.',
    'eli5': 'Explain things to a 5-year-old using simple words and concrete analogies. Keep answers under 3 sentences.',
}

for name, system_prompt in personas.items():
    response = client.messages.create(
        model='claude-haiku-4-5',
        max_tokens=512,
        system=system_prompt,
        messages=[{'role': 'user', 'content': user_question}],
    )
    text = next(b.text for b in response.content if b.type == 'text')
    print(f'\n--- {name.upper()} ---')
    print(text)

---
## 4. Multi-Turn Conversations

**The API is stateless.** Claude has no memory between calls — you must replay the whole conversation each turn. The price you pay: the input grows every turn (which is why **prompt caching** is a major topic later in the course).

### Rules
1. Messages alternate: `user` → `assistant` → `user` → `assistant` ...
2. The first message must be `user`.
3. To continue, append the assistant's response, then add a new `user` message.

Below is a minimal `Conversation` class that handles the bookkeeping:

In [ ]:
class Conversation:
    """Manages a multi-turn chat with Claude. Stateless API → we hold the history."""

    def __init__(self, client, model='claude-haiku-4-5', system=None):
        self.client = client
        self.model = model
        self.system = system
        self.messages = []  # the running history

    def send(self, user_text, max_tokens=1024):
        # 1. Append the user turn to history
        self.messages.append({'role': 'user', 'content': user_text})

        # 2. Send the FULL history (Claude doesn't remember prior calls)
        kwargs = {
            'model': self.model,
            'max_tokens': max_tokens,
            'messages': self.messages,
        }
        if self.system:
            kwargs['system'] = self.system
        response = self.client.messages.create(**kwargs)

        # 3. Extract the assistant's text and append it to history
        assistant_text = next(
            (b.text for b in response.content if b.type == 'text'), ''
        )
        self.messages.append({'role': 'assistant', 'content': assistant_text})
        return assistant_text


# Try it: Claude should remember earlier turns because we replay the full history
chat = Conversation(
    client,
    system='You are a friendly assistant. Keep responses under 2 sentences.'
)

print('USER: My name is Sam and I love astronomy.')
print('CLAUDE:', chat.send('My name is Sam and I love astronomy.'))
print()
print('USER: What hobby did I just mention?')
print('CLAUDE:', chat.send('What hobby did I just mention?'))
print()
print(f'-- history now has {len(chat.messages)} turns --')

---
## 5. Streaming — Token-by-Token Output

For chat UIs and long responses, you don't want to wait for the full response. **Streaming** gives you tokens as they're generated.

Use `client.messages.stream()` (a context manager) and iterate over `stream.text_stream`:

In [ ]:
import sys

with client.messages.stream(
    model='claude-haiku-4-5',
    max_tokens=512,
    messages=[{'role': 'user', 'content': 'Write a haiku about Python notebooks.'}],
) as stream:
    for chunk in stream.text_stream:
        print(chunk, end='', flush=True)

    # After streaming finishes, you can still access the full message
    final = stream.get_final_message()
    print(f'\n\n--- streamed {final.usage.output_tokens} output tokens ---')

### When to stream
- **Always stream** when `max_tokens > 16000` — non-streaming requests can hit SDK HTTP timeouts.
- Stream for any user-facing chat UI to feel responsive.
- Skip streaming for batch jobs where you only need the final result.

---
## 6. Model Selection — The Cost/Quality Tradeoff

| Model | Best for | Input $/1M | Output $/1M |
|---|---|---|---|
| `claude-opus-4-7` | Hardest reasoning, agentic work | $5.00 | $25.00 |
| `claude-sonnet-4-6` | Most production workloads | $3.00 | $15.00 |
| `claude-haiku-4-5` | Classification, simple Q&A | $1.00 | $5.00 |

Let's compare how the same prompt fares across tiers:

In [ ]:
import time

prompt = 'In exactly one sentence: why does sorting an already-sorted list with quicksort have O(n^2) worst-case time?'

for model in ['claude-haiku-4-5', 'claude-sonnet-4-6']:
    t0 = time.time()
    response = client.messages.create(
        model=model,
        max_tokens=512,
        messages=[{'role': 'user', 'content': prompt}],
    )
    elapsed = time.time() - t0
    text = next(b.text for b in response.content if b.type == 'text')
    tokens = response.usage.input_tokens + response.usage.output_tokens
    print(f'\n=== {model} ({elapsed:.2f}s, {tokens} tokens) ===')
    print(text)

---
## 7. Token Counting — Estimate Before You Spend

`client.messages.count_tokens()` gives you the input token count **without making a billed inference call**. Use it to:
- Verify a prompt fits in the context window
- Estimate cost before running a batch
- Decide whether to truncate or chunk

Note: this only counts **input** tokens (you can't predict output length without running the model).

In [ ]:
long_prompt = 'Explain the history of computer science.' * 50  # padded to be visible

result = client.messages.count_tokens(
    model='claude-haiku-4-5',
    messages=[{'role': 'user', 'content': long_prompt}],
)

input_cost_per_token = 1.00 / 1_000_000  # Haiku 4.5 input pricing
estimated_input_cost = result.input_tokens * input_cost_per_token

print(f'Input tokens: {result.input_tokens}')
print(f'Estimated input cost: ${estimated_input_cost:.6f}')

---
## What's Next?

The course covers many topics not in this starter notebook. Suggested next steps:

1. **Tool use** — let Claude call functions you define (calculator, database, web fetch)
2. **Prompt caching** — drop input cost ~90% by caching stable prefixes
3. **Structured outputs** — guarantee responses match a JSON schema
4. **Vision** — pass images and PDFs to Claude
5. **Extended thinking** — let Claude reason internally before answering hard problems
6. **Batches API** — 50% cost discount for non-latency-sensitive workloads
7. **MCP (Model Context Protocol)** — connect Claude to external tools and data sources

Each one is an additional parameter on `client.messages.create()` — the foundation you've built here scales directly. Try modifying the cells above to experiment, and refer back to the [course page](https://anthropic.skilljar.com/claude-with-the-anthropic-api/287725) for the next lessons.